# Notebook 04 — Inference Application
## Landmark Classification CNN · UCB MSc Data Science & AI

> **Business question:** Can the deployed model correctly identify landmarks in real-world images not seen during training?

**Phase 4 rubric targets:**
- `predict_landmarks(img_path, k)` function using TorchScript (Fase 4)
- Test on ≥4 personal images not from the dataset
- Written analysis of strengths and weaknesses

> ✅ **This notebook runs on PyCharm (CPU).** Inference is lightweight.

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO)

from src.config import EXTERNAL_DIR, MODELS_DIR, SEED
from src.utils import set_seed

set_seed(SEED)

# List available TorchScript models
scripted_models = sorted(MODELS_DIR.glob('*_scripted.pt'))
print('Available TorchScript models:')
for m in scripted_models:
    print(f'  {m.name}')

## 1. Select Best Model

Load the best-performing TorchScript model from Phase 3.
No model.py dependency — the scripted graph is self-contained.

In [ ]:
# ── Cell 2: Select best TorchScript model ───────────────────────
# Why TorchScript for inference:
# The .pt scripted file contains the full computation graph.
# No need to import model.py, config.py, or any src module.
# Deployable to any Python environment with only torch installed.

# Select the finetune model if available — highest expected accuracy
finetune_models = [m for m in scripted_models if 'finetune' in m.name]
BEST_MODEL_PATH = finetune_models[-1] if finetune_models else scripted_models[-1]
print(f'Using model: {BEST_MODEL_PATH.name}')

## 2. Run predict_landmarks on Personal Images

Place ≥4 personal images (JPG or PNG) in `data/external/` before running.
These must be images **not from the Google Landmarks dataset**.

In [ ]:
# ── Cell 3: Verify external images ─────────────────────────────
# Why verify before inference:
# A missing or corrupt image raises an unhelpful CUDA error during
# the forward pass. Checking existence here gives a clear error message.
images = (
    list(EXTERNAL_DIR.glob('*.jpg'))
    + list(EXTERNAL_DIR.glob('*.jpeg'))
    + list(EXTERNAL_DIR.glob('*.png'))
    + list(EXTERNAL_DIR.glob('*.webp'))
)
print(f'External images found: {len(images)}')
for img in images:
    print(f'  {img.name}')

assert len(images) >= 4, (
    f'Rubric requires >=4 personal images in data/external/. Found: {len(images)}'
)

In [ ]:
# ── Cell 4: Inference on all external images ────────────────────
# predict_and_display: runs inference + renders BI visualization
# (UCB palette bar chart alongside original image).
# Why display probability bars and not just the top-1 label:
# Production systems show top-3 suggestions — users can correct
# if the top-1 is wrong. Probability bars communicate model confidence.
from src.predictor import predict_and_display

all_predictions = {}

for img_path in images:
    print(f'\n=== {img_path.name} ===')
    preds = predict_and_display(
        img_path            = img_path,
        k                   = 3,
        scripted_model_path = BEST_MODEL_PATH,
    )
    all_predictions[img_path.name] = preds
    for rank, (name, prob) in enumerate(preds, 1):
        print(f'  {rank}. {name.replace("_", " ")}: {prob:.2%}')

## 3. Prediction Summary Table

Structured view of all predictions for the written analysis.

In [ ]:
# ── Cell 5: Prediction summary table ───────────────────────────
import pandas as pd

rows = []
for img_name, preds in all_predictions.items():
    for rank, (name, prob) in enumerate(preds, 1):
        rows.append({
            'Image'      : img_name,
            'Rank'       : rank,
            'Prediction' : name.replace('_', ' '),
            'Confidence' : f'{prob:.2%}',
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 4. Analysis — Strengths and Weaknesses

*(Fill based on your prediction results)*

### Strengths
- High confidence (>80%) on iconic, visually distinctive landmarks
- Top-3 predictions include the correct class even when Top-1 is wrong
- Robust to moderate changes in lighting and viewpoint (augmentation effect)

### Weaknesses
- Low confidence on landmarks not well-represented in training data
- Confuses architecturally similar structures (e.g. Gothic cathedrals)
- Performance degrades on heavily cropped or extreme-angle images

### Possible Improvements
- Ensemble: combine ResNet18 + ResNet50 predictions for borderline cases
- Test-Time Augmentation: average predictions over multiple crops
- Collect more images of weak classes (identified in Executive Report)

## Phase 4 — Checklist

| Check | Status |
|---|---|
| `predict_landmarks()` uses TorchScript | ✅ |
| ≥4 personal images tested | ✅ |
| Visual results displayed (BI bar chart) | ✅ |
| Prediction summary table | ✅ |
| Written analysis strengths/weaknesses | ✅ |

## Final Deliverables Checklist

| Deliverable | Status |
|---|---|
| GitHub repo public (no dataset) | ⬜ |
| All 4 notebooks executed with outputs | ⬜ |
| README.md with YouTube link | ⬜ |
| Video 3–5 min uploaded to YouTube | ⬜ |
| docs/informe_landmarks.pdf | ⬜ |
| git tag v1.0-entrega pushed | ⬜ |